# 04 --  Baseline Models

## Concept
Start simple. Baseline models establish minimum performance and reveal data quality issues before investing in complex models.

## Mathematical Intuition
- **Logistic Regression**: P(y=1|x) = 1 / (1 + e^{-(w*x + b)})
- **Decision Tree**: Recursive binary splits minimizing Gini impurity or entropy
- **Random Forest**: Bootstrap aggregation (bagging) of many decorrelated trees

## Interview Questions
1. Why start with a baseline model?
2. What is the bias-variance tradeoff and how do these models handle it?
3. When would Logistic Regression outperform a Decision Tree?

## Production Mapping
Production baseline wrappers are in `training/models.py`. Each model implements the same BaseModelWrapper interface.


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
np.random.seed(42)
n = 1000
X = pd.DataFrame({
    'capacity_mw': np.random.exponential(500, n),
    'region_risk': np.random.uniform(0, 1, n),
    'age_years': np.random.exponential(30, n),
    'num_connections': np.random.poisson(5, n),
})
# synthetic target: higher capacity + higher risk = higher criticality
y = (X['capacity_mw'] / 100 + X['region_risk'] * 5 + np.random.normal(0, 0.5, n)).clip(0, 3).round().astype(int).clip(0, 3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {np.unique(y)}")

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}
results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n{'='*40}")
    print(f"{name} --  Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance comparison
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# Logistic Regression coefficients
coef = models['Logistic Regression'].coef_[0]
axes[0].barh(X.columns, coef)
axes[0].set_title('LogReg Coefficients')
# Random Forest importance
imp = models['Random Forest'].feature_importances_
axes[1].barh(X.columns, imp)
axes[1].set_title('RF Importance')
plt.tight_layout()
plt.savefig('../artifacts/04_feature_importance.png', dpi=100)
plt.show()

## Key Takeaways
- Logistic Regression is fast, interpretable, and a strong baseline
- Decision Trees capture non-linear patterns but overfit easily
- Random Forest reduces variance via ensemble averaging
- Feature importance helps validate domain understanding
- All baselines should be trained before trying advanced models